In [1]:
import re
from pathlib import Path

import pandas as pd

In [7]:
paths = sorted(Path("output/finetune").rglob("eval_table_best.csv"))

tables = []
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-3]
    match = re.search(r"(full|lora|partial|scratch)_lr([-0-9e]+)", key)
    if not match:
        continue
    subset, lr = match.groups()
    table.insert(0, "subset", subset)
    tables.append(table)
table = pd.concat(tables, ignore_index=True)

In [8]:
summary = []
for _, group in table.groupby(["repr", "subset", "dataset"]):
    idx = group.query("split == 'validation'")["acc"].idxmax()
    lr = group.loc[idx, "lr"]
    summary.append(group.query(f"lr == {lr}"))
summary = pd.concat(summary, ignore_index=True)
summary = summary.loc[:, ["dataset", "repr", "subset", "lr", "epoch", "split", "acc"]]
summary = summary.sort_values(["dataset", "repr", "subset", "split"])
summary

,dataset,repr,subset,lr,epoch,split,acc
2,nsd_cococlip,patch,full,0.00003,6,test,0.298145
3,nsd_cococlip,patch,full,0.00003,6,testid,0.318296
0,nsd_cococlip,patch,full,0.00003,6,train,0.409201
1,nsd_cococlip,patch,full,0.00003,6,validation,0.295127
6,nsd_cococlip,patch,lora,0.00003,10,test,0.318553
7,nsd_cococlip,patch,lora,0.00003,10,testid,0.326200
4,nsd_cococlip,patch,lora,0.00003,10,train,0.424875
5,nsd_cococlip,patch,lora,0.00003,10,validation,0.297711
10,nsd_cococlip,patch,partial,0.00003,6,test,0.289054
11,nsd_cococlip,patch,partial,0.00003,6,testid,0.305764


In [12]:
print(summary.query("split == 'test'").loc[:, ["subset", "acc"]].to_markdown(index=False))

| subset   |      acc |
|:---------|---------:|
| full     | 0.298145 |
| lora     | 0.318553 |
| partial  | 0.289054 |


In [21]:
summary.query("repr == 'patch' and subset == 'qkv'")

,dataset,repr,subset,lr,epoch,split,acc
23,nsd_cococlip,patch,qkv,0.00001,17,test,0.325974
24,nsd_cococlip,patch,qkv,0.00001,17,testid,0.309620
21,nsd_cococlip,patch,qkv,0.00001,17,train,0.359599
22,nsd_cococlip,patch,qkv,0.00001,17,validation,0.301956
27,nsd_cococlip_subj01,patch,qkv,0.00001,18,test,0.445438
25,nsd_cococlip_subj01,patch,qkv,0.00001,18,train,0.672375
26,nsd_cococlip_subj01,patch,qkv,0.00001,18,validation,0.435185
